# Can a small model pass a test?

This notebook asks a fun question: **can a small browser model answer SAT-style multiple-choice questions?**

The point is not to prove that the model is brilliant. In fact, small browser models will often do poorly. That is part of the lesson. You will see where they make reasonable guesses, where they get confused, and how scoring works.

## Step 1: Get set up

This loads WebLLM, checks for WebGPU, and defines helpers for loading a chat model and asking questions in a consistent format.

In [ ]:
import js
import json
import random
import re
import pandas as pd
from pyodide.ffi import to_js, create_proxy
from IPython.display import display, Markdown

if getattr(js.navigator, "gpu", None):
    print("WebGPU is available. You can run models.")
else:
    print("WebGPU is not available here. Try the latest Chrome or Edge on a laptop.")

webllm = await js.eval("import('https://esm.run/@mlc-ai/web-llm')")
print("WebLLM loaded.")

engine = None
loaded_model = None


def js_obj(d):
    return to_js(d, dict_converter=js.Object.fromEntries)


def progress_bar():
    status = display("Starting...", display_id=True)

    def show(report):
        filled = int(report.progress * 30)
        bar = "#" * filled + "-" * (30 - filled)
        status.update(f"[{bar}] {report.progress:.0%}\n{report.text}")

    return status, create_proxy(show)


async def start_model(model_id):
    global engine
    status, progress = progress_bar()

    async def attempt(chat_options=None):
        global engine
        if engine is None:
            if chat_options:
                engine = await webllm.CreateMLCEngine(
                    model_id, js_obj({"initProgressCallback": progress}), js_obj(chat_options)
                )
            else:
                engine = await webllm.CreateMLCEngine(
                    model_id, js_obj({"initProgressCallback": progress})
                )
        else:
            engine.setInitProgressCallback(progress)
            if chat_options:
                await engine.reload(model_id, js_obj(chat_options))
            else:
                await engine.reload(model_id)

    try:
        await attempt()
    except Exception as error:
        if "WindowSizeConfigurationError" in str(error):
            status.update("Adjusting this model's settings and retrying...")
            await attempt({"sliding_window_size": -1})
        else:
            raise


def build_prompt(question_text, choices_text, passage_text=""):
    if passage_text:
        body = "Passage: " + passage_text + "\n\nQuestion: " + question_text
    else:
        body = "Question: " + question_text
    return (
        "Here is an SAT-style multiple-choice question.\n\n"
        + body
        + "\n\nChoices: " + choices_text + "\n\n"
        + "Think carefully. On the very last line of your response, write exactly: Final answer: X where X is A, B, C, or D."
    )


async def answer_question(question_text, choices_text, passage_text=""):
    if loaded_model is None:
        return {"error": "No model is loaded yet. Run Step 3."}
    messages = [
        {"role": "system", "content": "You are a careful tutor. Show your reasoning, then end with exactly one line in the form Final answer: X."},
        {"role": "user", "content": build_prompt(question_text, choices_text, passage_text)},
    ]
    request = {
        "messages": messages,
        "temperature": 0.0,
        "seed": 7,
        "max_tokens": 300,
        "frequency_penalty": 0.5,
    }
    reply = await engine.chat.completions.create(js_obj(request))
    result = reply.to_py()
    text = result['choices'][0]['message']['content']
    match = re.search(r'[Ff]inal answer:\s*([A-D])', text)
    result['parsed_answer'] = match.group(1).upper() if match else ''
    return result

## Step 2: Choose and load a model

A small model will make the experiment more honest. It may score poorly, but that is the point.

In [ ]:
MODEL_ID = "gemma3-1b-it-q4f16_1-MLC"
print("Model to load: " + MODEL_ID)

In [ ]:
await start_model(MODEL_ID)
loaded_model = MODEL_ID
print(loaded_model + " is loaded and ready.")

## Step 3: Warm up with one question

We start with one simple question so you can see the prompt format and the answer parser.

In [ ]:
warmup_question = "If 3x + 5 = 14, what is the value of x?"
warmup_choices = "A) 2\nB) 3\nC) 4\nD) 5"

warmup = await answer_question(warmup_question, warmup_choices)
if 'error' in warmup:
    print(warmup['error'])
else:
    print(warmup['choices'][0]['message']['content'])
    print()
    print("Parsed answer:", warmup['parsed_answer'])

## Step 4: Load a bundled PineSAT question bank

This notebook now comes with a **bundled dataset of 100 PineSAT questions** stored in `data/pinesat_100_questions.json`. That makes the notebook much more useful even if the old PineSAT website changes or disappears.

The file is a saved snapshot of 100 questions from the PineSAT question bank, with 50 English questions and 50 Math questions. It also keeps the difficulty, domain, and answer explanation for each item.


In [ ]:
DATASET_PATH = "data/pinesat_100_questions.json"

with open(DATASET_PATH, "r", encoding="utf-8") as f:
    dataset = json.load(f)

QUESTION_BANK = dataset["questions"]
summary = pd.DataFrame(QUESTION_BANK)
print("Loaded", len(QUESTION_BANK), "PineSAT questions from", DATASET_PATH)
print("Source archive:", dataset["source_archive"])
print()
print(summary.groupby(["section", "difficulty"]).size())
print()
summary[["section", "difficulty", "domain", "question", "answer"]].head(12)


## Step 5: Choose a section, difficulty, and set size

Because the models run on your machine, it is still better to start small even though the bundled bank has 100 questions. Two to five at a time is a good first test.


In [ ]:
SECTION = "English"
DIFFICULTY = "Easy"
NUM_QUESTIONS = 2

print("Section:    " + SECTION)
print("Difficulty: " + DIFFICULTY)
print("Questions:  " + str(NUM_QUESTIONS))

## Step 6: Build the practice set

This filters the question bank and picks a small sample.

In [ ]:
filtered = [
    item for item in QUESTION_BANK
    if item['section'] == SECTION and item['difficulty'] == DIFFICULTY
]

if not filtered:
    print("No questions match those settings. Change SECTION or DIFFICULTY.")
    practice_set = []
else:
    practice_set = filtered[:]
    random.shuffle(practice_set)
    practice_set = practice_set[:NUM_QUESTIONS]
    print("Built a practice set of", len(practice_set), "questions.")
    for i, item in enumerate(practice_set, start=1):
        print()
        print("Q" + str(i) + ":", item['question'])
        if item['passage']:
            print("Passage:", item['passage'])
        print("Choices:", item['choices'])
        print("Correct answer:", item['answer'])

## Step 7: Let the model take the mini test

The next cell asks each question, parses the final answer letter, and saves the results.

In [ ]:
results = []
for item in practice_set:
    reply = await answer_question(item['question'], item['choices'], item['passage'])
    if 'error' in reply:
        print(reply['error'])
        break
    model_text = reply['choices'][0]['message']['content']
    model_guess = reply['parsed_answer']
    is_correct = model_guess == item['answer']
    results.append({
        'section': item['section'],
        'difficulty': item['difficulty'],
        'domain': item['domain'],
        'question': item['question'],
        'correct_answer': item['answer'],
        'model_guess': model_guess,
        'correct': is_correct,
        'raw_reply': model_text,
    })

results_table = pd.DataFrame(results)
results_table[['section', 'difficulty', 'domain', 'correct_answer', 'model_guess', 'correct']]


## Step 8: Read the model's full replies

Sometimes the most interesting part is not whether the answer is right, but how the model explained itself.

In [ ]:
if len(results) == 0:
    print("No results yet. Run Step 7 first.")
else:
    for i, row in enumerate(results, start=1):
        print("=" * 70)
        print("Question", i)
        print(row['question'])
        print()
        print(row['raw_reply'])
        print()
        print("Parsed answer:", row['model_guess'])
        print("Correct answer:", row['correct_answer'])
        print("Correct?:", row['correct'])

## Step 9: Score the model

Now we will make two summaries:

1. A **scoring table** that shows how the model did overall, by section, by difficulty, and by domain.
2. A **model report card** that turns those numbers into a short plain-language summary.

Small browser models often miss several questions. That is useful evidence, not a failure of the notebook.


In [ ]:
if len(results) == 0:
    print("No results yet. Run Step 7 first.")
else:
    results_table = pd.DataFrame(results)
    scored_questions = results_table[[
        'section', 'difficulty', 'domain', 'question', 'correct_answer', 'model_guess', 'correct'
    ]].copy()

    display(scored_questions)

    score_rows = [
        {
            'group': 'Overall',
            'label': 'All questions',
            'correct_answers': int(results_table['correct'].sum()),
            'total_questions': len(results_table),
        }
    ]

    for section, group in results_table.groupby('section', sort=True):
        score_rows.append({
            'group': 'Section',
            'label': section,
            'correct_answers': int(group['correct'].sum()),
            'total_questions': len(group),
        })

    for difficulty, group in results_table.groupby('difficulty', sort=True):
        score_rows.append({
            'group': 'Difficulty',
            'label': difficulty,
            'correct_answers': int(group['correct'].sum()),
            'total_questions': len(group),
        })

    for domain, group in results_table.groupby('domain', sort=True):
        score_rows.append({
            'group': 'Domain',
            'label': domain,
            'correct_answers': int(group['correct'].sum()),
            'total_questions': len(group),
        })

    score_table = pd.DataFrame(score_rows)
    score_table['accuracy'] = (100 * score_table['correct_answers'] / score_table['total_questions']).round(1)
    display(score_table)

    overall_row = score_table.iloc[0]
    overall_accuracy = float(overall_row['accuracy'])
    blank_answers = int((results_table['model_guess'] == '').sum())

    if overall_accuracy >= 90:
        letter_grade = 'A'
        verdict = 'This model was unusually strong on this mini test.'
    elif overall_accuracy >= 80:
        letter_grade = 'B'
        verdict = 'This model did well, with some noticeable mistakes.'
    elif overall_accuracy >= 70:
        letter_grade = 'C'
        verdict = 'This model was mixed: some solid answers, some clear misses.'
    elif overall_accuracy >= 60:
        letter_grade = 'D'
        verdict = 'This model struggled and would need careful checking by a human.'
    else:
        letter_grade = 'F'
        verdict = 'This model had a hard time with this question set.'

    detail_rows = score_table[score_table['group'] != 'Overall'].copy()
    detail_rows['sort_key'] = detail_rows['accuracy'] + (detail_rows['total_questions'] / 1000)
    strongest_row = detail_rows.sort_values(['sort_key', 'total_questions'], ascending=False).iloc[0]
    weakest_row = detail_rows.sort_values(['accuracy', 'total_questions'], ascending=[True, False]).iloc[0]

    report_lines = [
        '## Model report card',
        '',
        f'- **Model:** {loaded_model}',
        f'- **Questions graded:** {int(overall_row["total_questions"])}',
        f'- **Score:** {int(overall_row["correct_answers"])} out of {int(overall_row["total_questions"])}',
        f'- **Accuracy:** {overall_accuracy:.1f}%',
        f'- **Letter grade:** {letter_grade}',
        f'- **Blank or unparsed answers:** {blank_answers}',
        f'- **Strongest area in this run:** {strongest_row["group"]} - {strongest_row["label"]} ({strongest_row["accuracy"]:.1f}%)',
        f'- **Weakest area in this run:** {weakest_row["group"]} - {weakest_row["label"]} ({weakest_row["accuracy"]:.1f}%)',
        '',
        verdict,
    ]
    display(Markdown('\n'.join(report_lines)))


## Step 10: Optional: explore domains and explanations in the bundled bank

The PineSAT snapshot includes a **domain** for each question, such as Algebra or Information and Ideas, plus a written explanation for the correct answer. This lets you study not just scores, but also what kinds of questions are in the bank.


In [ ]:
DOMAIN = ""

domain_rows = pd.DataFrame(QUESTION_BANK)
if DOMAIN:
    domain_rows = domain_rows[domain_rows["domain"] == DOMAIN]

print("Questions shown:", len(domain_rows))
domain_rows[["section", "difficulty", "domain", "question", "explanation"]].head(10)


## Your turn

Try a different section, difficulty, or number of questions. Then switch to another small model and compare the scores. Which mistakes seem random, and which mistakes reveal a real weakness in the model's reasoning or reading? You can also filter the bundled bank by domain and see whether some topics are harder than others.
